# Resultats complementaris de les automatitzacions sobre 180 sèries

Aquest notebook resumeix els resultats obtinguts amb les dues automatitzacions del TFG: els models SARIMA i els models híbrids SARIMA-NNAR. L'objectiu és obtenir taules per complementar l'anàlisi detallada de les sèries representatives.

Els resultats es comparen només en aquelles sèries on tots dos models s'han ajustat correctament.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

SARIMA_PATH = Path("results_sarima.csv")
HYBRID_PATH = Path("results_hybrid.csv")

sarima = pd.read_csv(SARIMA_PATH)
hybrid = pd.read_csv(HYBRID_PATH)

resum_carrega = pd.DataFrame({
    "Fitxer": [SARIMA_PATH.name, HYBRID_PATH.name],
    "Nombre de files": [len(sarima), len(hybrid)],
    "Nombre de columnes": [sarima.shape[1], hybrid.shape[1]]
})

resum_carrega

,Fitxer,Nombre de files,Nombre de columnes
0,results_sarima.csv,180,19
1,results_hybrid.csv,174,29


## Unió de sèries comparables

Primer es conserven només les sèries amb `status == "ok"`. Després s'uneixen els dos resultats utilitzant com a clau el grup d'edat, el diagnòstic i la regió.

In [2]:
KEYS = ["age", "diag", "region"]

SARIMA_RENAME = {
    "mae_test": "sarima_mae",
    "rmse_test": "sarima_rmse",
    "cov_total": "sarima_cov_total",
    "cov_covid": "sarima_cov_covid",
    "cov_post": "sarima_cov_post",
}

HYBRID_METRICS = [
    "hybrid_mae",
    "hybrid_rmse",
    "hybrid_cov_total",
    "hybrid_cov_covid",
    "hybrid_cov_post",
]

required_columns = {
    "SARIMA": KEYS + ["status"] + list(SARIMA_RENAME.keys()),
    "Híbrid": KEYS + ["status"] + HYBRID_METRICS,
}

missing_columns = {
    name: sorted(set(cols) - set(data.columns))
    for name, cols, data in [
        ("SARIMA", required_columns["SARIMA"], sarima),
        ("Híbrid", required_columns["Híbrid"], hybrid),
    ]
}

if any(missing_columns.values()):
    raise ValueError(f"Falten columnes obligatòries: {missing_columns}")

sarima_ok = sarima.loc[sarima["status"] == "ok"].copy()
hybrid_ok = hybrid.loc[hybrid["status"] == "ok"].copy()

for name, data in {"SARIMA": sarima_ok, "Híbrid": hybrid_ok}.items():
    if data.duplicated(KEYS).any():
        duplicated = data.loc[data.duplicated(KEYS, keep=False), KEYS]
        raise ValueError(f"Hi ha claus duplicades a {name}:\n{duplicated}")

sarima_comp = sarima_ok[KEYS + list(SARIMA_RENAME.keys())].rename(columns=SARIMA_RENAME)
hybrid_comp = hybrid_ok[KEYS + HYBRID_METRICS].copy()

df = hybrid_comp.merge(sarima_comp, on=KEYS, how="inner", validate="one_to_one")

resum_ajust = pd.DataFrame({
    "Indicador": [
        "Sèries SARIMA correctes",
        "Sèries híbrides correctes",
        "Sèries comparables SARIMA vs híbrid",
    ],
    "Valor": [len(sarima_ok), len(hybrid_ok), len(df)]
})

resum_ajust

,Indicador,Valor
0,Sèries SARIMA correctes,174
1,Sèries híbrides correctes,160
2,Sèries comparables SARIMA vs híbrid,160


## Càlcul de millores del model híbrid

La millora es calcula com la diferència entre l'error del SARIMA i l'error del model híbrid. 

In [3]:
for metric in ["mae", "rmse"]:
    sarima_col = f"sarima_{metric}"
    hybrid_col = f"hybrid_{metric}"
    delta_col = f"delta_{metric}"
    pct_col = f"delta_{metric}_pct"
    improvement_col = f"millora_{metric}"

    df[delta_col] = df[sarima_col] - df[hybrid_col]
    df[pct_col] = np.where(
        df[sarima_col] != 0,
        df[delta_col] / df[sarima_col] * 100,
        np.nan,
    )
    df[improvement_col] = df[delta_col] > 0

Les taules següents resumeixen la comparació dels errors i de la cobertura dels intervals de predicció, diferenciant el període COVID i el període posterior.

In [4]:
resum_errors = pd.DataFrame({
    "Mètrica": [
        "MAE mediana SARIMA",
        "MAE mediana híbrid",
        "Millora mediana MAE (%)",
        "RMSE mediana SARIMA",
        "RMSE mediana híbrid",
        "Millora mediana RMSE (%)",
        "Sèries on millora MAE (%)",
        "Sèries on millora RMSE (%)",
    ],
    "Valor": [
        round(df["sarima_mae"].median(), 3),
        round(df["hybrid_mae"].median(), 3),
        round(df["delta_mae_pct"].median(), 2),
        round(df["sarima_rmse"].median(), 3),
        round(df["hybrid_rmse"].median(), 3),
        round(df["delta_rmse_pct"].median(), 2),
        round(df["millora_mae"].mean() * 100, 1),
        round(df["millora_rmse"].mean() * 100, 1),
    ]
})

resum_cobertura = pd.DataFrame({
    "Període": ["Total", "COVID", "Post-COVID"],
    "SARIMA mitjana": [
        round(df["sarima_cov_total"].mean(), 3),
        round(df["sarima_cov_covid"].mean(), 3),
        round(df["sarima_cov_post"].mean(), 3),
    ],
    "SARIMA mediana": [
        round(df["sarima_cov_total"].median(), 3),
        round(df["sarima_cov_covid"].median(), 3),
        round(df["sarima_cov_post"].median(), 3),
    ],
    "Híbrid mitjana": [
        round(df["hybrid_cov_total"].mean(), 3),
        round(df["hybrid_cov_covid"].mean(), 3),
        round(df["hybrid_cov_post"].mean(), 3),
    ],
    "Híbrid mediana": [
        round(df["hybrid_cov_total"].median(), 3),
        round(df["hybrid_cov_covid"].median(), 3),
        round(df["hybrid_cov_post"].median(), 3),
    ],
})

resum_cobertura_baixa = pd.DataFrame({
    "Indicador": [
        "SARIMA: cobertura COVID < 50%",
        "SARIMA: cobertura COVID < 70%",
        "SARIMA: cobertura COVID < 80%",
        "Híbrid: cobertura COVID < 50%",
        "Híbrid: cobertura COVID < 70%",
        "Híbrid: cobertura COVID < 80%",
    ],
    "Nombre de sèries": [
        (df["sarima_cov_covid"] < 0.5).sum(),
        (df["sarima_cov_covid"] < 0.7).sum(),
        (df["sarima_cov_covid"] < 0.8).sum(),
        (df["hybrid_cov_covid"] < 0.5).sum(),
        (df["hybrid_cov_covid"] < 0.7).sum(),
        (df["hybrid_cov_covid"] < 0.8).sum(),
    ],
    "Percentatge": [
        round((df["sarima_cov_covid"] < 0.5).mean() * 100, 1),
        round((df["sarima_cov_covid"] < 0.7).mean() * 100, 1),
        round((df["sarima_cov_covid"] < 0.8).mean() * 100, 1),
        round((df["hybrid_cov_covid"] < 0.5).mean() * 100, 1),
        round((df["hybrid_cov_covid"] < 0.7).mean() * 100, 1),
        round((df["hybrid_cov_covid"] < 0.8).mean() * 100, 1),
    ],
})

df["covid_menor_que_post_sarima"] = df["sarima_cov_covid"] < df["sarima_cov_post"]
df["covid_menor_que_post_hybrid"] = df["hybrid_cov_covid"] < df["hybrid_cov_post"]

resum_covid = pd.DataFrame({
    "Model": ["SARIMA", "SARIMA-NNAR"],
    "Sèries amb cobertura COVID inferior a post-COVID": [
        df["covid_menor_que_post_sarima"].sum(),
        df["covid_menor_que_post_hybrid"].sum(),
    ],
    "Percentatge": [
        round(df["covid_menor_que_post_sarima"].mean() * 100, 1),
        round(df["covid_menor_que_post_hybrid"].mean() * 100, 1),
    ],
})

display(resum_errors)
display(resum_cobertura)
display(resum_cobertura_baixa)
display(resum_covid)

,Mètrica,Valor
0,MAE mediana SARIMA,350.806
1,MAE mediana híbrid,353.126
2,Millora mediana MAE (%),0.120
3,RMSE mediana SARIMA,464.282
4,RMSE mediana híbrid,469.155
5,Millora mediana RMSE (%),0.250
6,Sèries on millora MAE (%),53.800
7,Sèries on millora RMSE (%),54.400


,Període,SARIMA mitjana,SARIMA mediana,Híbrid mitjana,Híbrid mediana
0,Total,0.825,0.873,0.859,0.906
1,COVID,0.727,0.837,0.770,0.889
2,Post-COVID,0.891,0.923,0.918,0.942


,Indicador,Nombre de sèries,Percentatge
0,SARIMA: cobertura COVID < 50%,30,18.8
1,SARIMA: cobertura COVID < 70%,57,35.6
2,SARIMA: cobertura COVID < 80%,72,45.0
3,Híbrid: cobertura COVID < 50%,26,16.2
4,Híbrid: cobertura COVID < 70%,48,30.0
5,Híbrid: cobertura COVID < 80%,63,39.4


,Model,Sèries amb cobertura COVID inferior a post-COVID,Percentatge
0,SARIMA,119,74.4
1,SARIMA-NNAR,119,74.4


In [5]:
taula_tfg = pd.DataFrame({
    "Resultat": [
        "Sèries SARIMA correctes",
        "Sèries híbrides correctes",
        "Sèries comparables",
        "Millora MAE del model híbrid",
        "Millora RMSE del model híbrid",
        "Millora mediana MAE",
        "Millora mediana RMSE",
        "Cobertura mediana COVID SARIMA",
        "Cobertura mediana COVID híbrid",
        "Cobertura mediana post-COVID SARIMA",
        "Cobertura mediana post-COVID híbrid",
        "Cobertura COVID inferior a post-COVID, SARIMA",
        "Cobertura COVID inferior a post-COVID, híbrid",
    ],
    "Valor": [
        len(sarima_ok),
        len(hybrid_ok),
        len(df),
        f"{df['millora_mae'].mean() * 100:.1f}%",
        f"{df['millora_rmse'].mean() * 100:.1f}%",
        f"{df['delta_mae_pct'].median():.2f}%",
        f"{df['delta_rmse_pct'].median():.2f}%",
        f"{df['sarima_cov_covid'].median() * 100:.1f}%",
        f"{df['hybrid_cov_covid'].median() * 100:.1f}%",
        f"{df['sarima_cov_post'].median() * 100:.1f}%",
        f"{df['hybrid_cov_post'].median() * 100:.1f}%",
        f"{df['covid_menor_que_post_sarima'].mean() * 100:.1f}%",
        f"{df['covid_menor_que_post_hybrid'].mean() * 100:.1f}%",
    ]
})

taula_tfg

,Resultat,Valor
0,Sèries SARIMA correctes,174
1,Sèries híbrides correctes,160
2,Sèries comparables,160
3,Millora MAE del model híbrid,53.8%
4,Millora RMSE del model híbrid,54.4%
5,Millora mediana MAE,0.12%
6,Millora mediana RMSE,0.25%
7,Cobertura mediana COVID SARIMA,83.7%
8,Cobertura mediana COVID híbrid,88.9%
9,Cobertura mediana post-COVID SARIMA,92.3%


## Rànquings de sèries

In [6]:
ranking_cols = KEYS + [
    "sarima_mae",
    "hybrid_mae",
    "delta_mae_pct",
    "sarima_rmse",
    "hybrid_rmse",
    "delta_rmse_pct",
    "sarima_cov_covid",
    "hybrid_cov_covid",
]

ranking_millora_RMSE = (
    df.sort_values("delta_rmse_pct", ascending=False)
      .loc[:, ranking_cols]
      .head(10)
)

ranking_empitjora_RMSE = (
    df.sort_values("delta_rmse_pct", ascending=True)
      .loc[:, ranking_cols]
      .head(10)
)

print("Sèries on més millora el RMSE amb el model híbrid")
display(ranking_millora_RMSE)

print("Sèries on més empitjora el RMSE amb el model híbrid")
display(ranking_empitjora_RMSE)

Sèries on més millora el RMSE amb el model híbrid


,age,diag,region,sarima_mae,hybrid_mae,delta_mae_pct,sarima_rmse,hybrid_rmse,delta_rmse_pct,sarima_cov_covid,hybrid_cov_covid
154,15+,PneumÃ²nia,Camp de Tarragona,48.080793,44.301932,7.859398,61.394517,57.152466,6.909495,0.942308,0.980769
99,15+,Bronquiolitis,Terres de l'Ebre,70.297944,63.307264,9.944359,89.334436,84.426402,5.494000,0.942308,0.971154
142,15+,Impetigen,Barcelona Metropolitana Nord,52.670102,49.499368,6.019989,70.441089,66.645875,5.387784,0.980769,1.000000
30,0-14,Faringoamigdalitis,Alt Pirineu i Aran,704.708760,661.810992,6.087305,813.241314,770.032595,5.313148,0.576923,0.846154
7,0-14,Altres IRA,Lleida,2559.154692,2371.028931,7.351090,3453.296169,3281.297296,4.980716,0.586538,0.634615
8,0-14,Altres IRA,PenedÃ¨s,2422.100305,2263.211310,6.559968,3272.646144,3110.168733,4.964711,0.615385,0.663462
9,0-14,Altres IRA,Terres de l'Ebre,1727.910632,1643.855859,4.864532,2308.887974,2204.644118,4.514894,0.653846,0.634615
5,0-14,Altres IRA,Catalunya Central,3631.944445,3488.235470,3.956805,4837.303049,4642.148027,4.034377,0.548077,0.596154
111,15+,Faringoamigdalitis,Barcelona Ciutat,722.192082,684.681601,5.193976,865.358152,830.511523,4.026845,0.250000,0.288462
156,15+,PneumÃ²nia,Girona,49.282479,47.342689,3.936062,64.413540,61.834477,4.003915,0.875000,0.951923


Sèries on més empitjora el RMSE amb el model híbrid


,age,diag,region,sarima_mae,hybrid_mae,delta_mae_pct,sarima_rmse,hybrid_rmse,delta_rmse_pct,sarima_cov_covid,hybrid_cov_covid
95,15+,Bronquiolitis,Catalunya Central,75.614822,90.282224,-19.397522,93.963677,110.443374,-17.538369,0.951923,0.990385
70,0-14,PneumÃ²nia,Alt Pirineu i Aran,562.797304,651.715936,-15.799406,745.664859,825.518636,-10.709071,0.932692,0.961538
15,0-14,Bronquiolitis,Catalunya Central,1095.118421,1261.775128,-15.218145,1573.100750,1715.550693,-9.055360,0.932692,0.951923
10,0-14,Bronquiolitis,Alt Pirineu i Aran,1258.095078,1316.314534,-4.627588,1814.043825,1949.512511,-7.467774,0.942308,0.951923
124,15+,Faringoamigdalitis estreptocÃ²ccica,Camp de Tarragona,93.563348,99.280873,-6.110859,114.414744,122.418625,-6.995498,0.442308,0.625000
29,0-14,Escarlatina,Terres de l'Ebre,230.642170,271.991815,-17.928050,339.948254,362.897180,-6.750712,0.961538,0.971154
40,0-14,Faringoamigdalitis estreptocÃ²ccica,Alt Pirineu i Aran,436.145315,474.083945,-8.698621,581.118000,619.087891,-6.533938,0.903846,0.913462
92,15+,Bronquiolitis,Barcelona Metropolitana Nord,31.545499,35.724884,-13.248752,45.033339,47.686530,-5.891614,0.942308,0.971154
147,15+,Impetigen,Lleida,87.818654,101.625881,-15.722431,136.038781,143.806805,-5.710154,0.942308,0.951923
110,15+,Faringoamigdalitis,Alt Pirineu i Aran,119.151043,127.551009,-7.049847,141.004791,149.040267,-5.698725,0.701923,0.855769
